# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [ ]:
with open('aaaa.txt', 'r', encoding='utf-8') as f:
    data = f.read()

print(data)


aaaa


In [ ]:
!nvidia-smi

Thu Apr  2 05:10:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             42W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import os
os.getcwd()

'/content'

In [ ]:
!pip install git+https://github.com/huggingface/transformers accelerate
!pip install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip install "peft>=0.14.0" "bitsandbytes>=0.46.1" datasets pillow pandas sentencepiece einops --upgrade
!pip install flash-attn --no-build-isolation

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-pustrqw6
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-pustrqw6
  Resolved https://github.com/huggingface/transformers to commit f38d6639fa6b82a401f4e2ea7fef1a3eb550c1a6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
!pip install flash-attn --no-build-isolation

  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user


In [ ]:
!pip install -v flash-attn --no-build-isolation

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Running command python setup.py egg_info
  /usr/local/lib/python3.12/dist-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
    warn(
  /usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
  !!

          ********************************************************************************
          Requirements should be satisfied by a PEP 517 installer.
          If you are using pip, you can try `pip install --use-pep517`.
          ********************************************************************************

  !!
    dist.fetch_build_eggs(dist.s

In [41]:
import torch
import transformers
import accelerate
import peft
import bitsandbytes

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("cuda available:", torch.cuda.is_available())

torch: 2.10.0+cu128
transformers: 5.5.0.dev0
accelerate: 1.13.0
peft: 0.18.1
bitsandbytes: 0.49.2
cuda available: True


In [ ]:
import flash_attn
print("flash-attn 설치됨")

ModuleNotFoundError: No module named 'flash_attn'

# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [42]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
import os
import json
import math
import torch
import pandas as pd

SAVE_ROOT = "/content/drive/MyDrive/recycle_vqa_qwen3"
BEST_DIR = os.path.join(SAVE_ROOT, "best_model")
LAST_DIR = os.path.join(SAVE_ROOT, "last_checkpoint")
STEP_DIR = os.path.join(SAVE_ROOT, "step_checkpoints")

os.makedirs(SAVE_ROOT, exist_ok=True)
os.makedirs(BEST_DIR, exist_ok=True)
os.makedirs(LAST_DIR, exist_ok=True)
os.makedirs(STEP_DIR, exist_ok=True)

In [ ]:
# 압축 해제
!unzip "/content/2026-ssafy-15-2-ai.zip" -d "/content/"

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/train/train_0074.jpg  
  inflating: /content/train/train_0075.jpg  
  inflating: /content/train/train_0076.jpg  
  inflating: /content/train/train_0077.jpg  
  inflating: /content/train/train_0078.jpg  
  inflating: /content/train/train_0079.jpg  
  inflating: /content/train/train_0080.jpg  
  inflating: /content/train/train_0081.jpg  
  inflating: /content/train/train_0082.jpg  
  inflating: /content/train/train_0083.jpg  
  inflating: /content/train/train_0084.jpg  
  inflating: /content/train/train_0085.jpg  
  inflating: /content/train/train_0086.jpg  
  inflating: /content/train/train_0087.jpg  
  inflating: /content/train/train_0088.jpg  
  inflating: /content/train/train_0089.jpg  
  inflating: /content/train/train_0090.jpg  
  inflating: /content/train/train_0091.jpg  
  inflating: /content/train/train_0092.jpg  
  inflating: /content/train/train_0093.jpg  
  inflating: /content/train/train_0094.jpg  
  inflating: /conte

# 라이브러리, 데이터, 설정 - 수정 완료

In [45]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from sklearn.model_selection import train_test_split

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
IMAGE_SIZE = 448
MAX_NEW_TOKENS = 4
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

# 학습데이터 200개만 추출
# train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

Device: cuda


# 모델, Processor - 수정 완료

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [46]:
USE_QLORA = True   # 먼저 True로 시작
USE_FLASH_ATTN = False

if USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
else:
    bnb_config = None

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
)

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

if USE_QLORA:
    base_model = prepare_model_for_kbit_training(base_model)

base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [47]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# Custom Dataset, Collator - 수정 완료

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [48]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text}
            ]}
        ]

        gold_text = None
        if self.train:
            gold_letter = str(row["answer"]).strip().lower()
            gold_map = {
                "a": a,
                "b": b,
                "c": c,
                "d": d,
            }
            gold_text = gold_map[gold_letter]
            messages.append({
                "role": "assistant",
                "content": [{"type": "text", "text": gold_text}]
            })

        return {
            "messages": messages,
            "image": img,
            "gold_text": gold_text
        }


@dataclass
class TrainCollator:
    processor: Any

    def __call__(self, batch):
        input_ids_list = []
        labels_list = []
        attention_masks_list = []
        pixel_values_list = []
        image_grid_thw_list = []
        mm_token_type_ids_list = []

        pad_id = self.processor.tokenizer.pad_token_id

        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            full_text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )

            prompt_messages = messages[:-1]
            prompt_text = self.processor.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )

            full_enc = self.processor(
                text=[full_text],
                images=[img],
                return_tensors="pt"
            )

            prompt_enc = self.processor(
                text=[prompt_text],
                images=[img],
                return_tensors="pt"
            )

            input_ids = full_enc["input_ids"][0]
            attention_mask = full_enc["attention_mask"][0]
            pixel_values = full_enc["pixel_values"][0]
            image_grid_thw = full_enc["image_grid_thw"][0]
            mm_token_type_ids = full_enc["mm_token_type_ids"][0]

            prompt_len = prompt_enc["input_ids"].shape[1]

            labels = input_ids.clone()
            labels[:prompt_len] = -100

            input_ids_list.append(input_ids)
            labels_list.append(labels)
            attention_masks_list.append(attention_mask)
            pixel_values_list.append(pixel_values)
            image_grid_thw_list.append(image_grid_thw)
            mm_token_type_ids_list.append(mm_token_type_ids)

        max_len = max(x.size(0) for x in input_ids_list)

        batch_input_ids = []
        batch_labels = []
        batch_attention_masks = []
        batch_mm_token_type_ids = []

        for input_ids, labels, attention_mask, mm_token_type_ids in zip(
            input_ids_list, labels_list, attention_masks_list, mm_token_type_ids_list
        ):
            pad_len = max_len - input_ids.size(0)

            batch_input_ids.append(
                torch.cat([input_ids, torch.full((pad_len,), pad_id, dtype=input_ids.dtype)])
            )
            batch_labels.append(
                torch.cat([labels, torch.full((pad_len,), -100, dtype=labels.dtype)])
            )
            batch_attention_masks.append(
                torch.cat([attention_mask, torch.zeros(pad_len, dtype=attention_mask.dtype)])
            )
            batch_mm_token_type_ids.append(
                torch.cat([mm_token_type_ids, torch.zeros(pad_len, dtype=mm_token_type_ids.dtype)])
            )

        batch_dict = {
            "input_ids": torch.stack(batch_input_ids),
            "labels": torch.stack(batch_labels),
            "attention_mask": torch.stack(batch_attention_masks),
            "pixel_values": torch.stack(pixel_values_list),
            "image_grid_thw": torch.stack(image_grid_thw_list),
            "mm_token_type_ids": torch.stack(batch_mm_token_type_ids),
        }

        return batch_dict

# DataLoader - 수정 완료

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [50]:
def make_qtype(q):
    q = str(q)
    if "몇 개" in q or "개수" in q:
        return "count"
    if "색" in q:
        return "color"
    if "재질" in q:
        return "material"
    if "모양" in q:
        return "shape"
    return "object"

train_df["qtype"] = train_df["question"].apply(make_qtype)

train_subset, valid_subset = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SEED,
    shuffle=True,
    stratify=train_df["qtype"]
)

train_subset = train_subset.reset_index(drop=True)
valid_subset = valid_subset.reset_index(drop=True)

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=TrainCollator(processor),
    num_workers=2,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=TrainCollator(processor),
    num_workers=2,
    pin_memory=True,
)

In [51]:
def save_log(logs, save_path):
    df = pd.DataFrame(logs)
    df.to_csv(save_path, index=False)

def save_training_state(save_dir, epoch, global_step, best_val_acc, logs):
    state = {
        "epoch": epoch,
        "global_step": global_step,
        "best_val_acc": best_val_acc,
        "logs": logs,
    }
    with open(os.path.join(save_dir, "training_state.json"), "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

def save_checkpoint(
    save_dir,
    model,
    processor,
    optimizer,
    scheduler,
    epoch,
    global_step,
    best_val_acc,
    logs
):
    os.makedirs(save_dir, exist_ok=True)

    model.save_pretrained(save_dir)
    processor.save_pretrained(save_dir)

    torch.save({
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "epoch": epoch,
        "global_step": global_step,
        "best_val_acc": best_val_acc,
        "logs": logs,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }, os.path.join(save_dir, "trainer_state.pt"))

    save_training_state(save_dir, epoch, global_step, best_val_acc, logs)

def load_checkpoint_if_exists(save_dir, model, optimizer=None, scheduler=None):
    trainer_state_path = os.path.join(save_dir, "trainer_state.pt")

    if not os.path.exists(trainer_state_path):
        print("체크포인트 없음. 처음부터 학습 시작.")
        return 0, 0, -1.0, []

    print(f"체크포인트 로드: {save_dir}")
    state = torch.load(trainer_state_path, map_location="cpu")

    if optimizer is not None and state.get("optimizer") is not None:
        optimizer.load_state_dict(state["optimizer"])

    if scheduler is not None and state.get("scheduler") is not None:
        scheduler.load_state_dict(state["scheduler"])

    start_epoch = state.get("epoch", 0)
    global_step = state.get("global_step", 0)
    best_val_acc = state.get("best_val_acc", -1.0)
    logs = state.get("logs", [])

    return start_epoch, global_step, best_val_acc, logs

In [52]:
from tqdm.auto import tqdm

def evaluate_val_loss(model, valid_loader, device):
    model.eval()
    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for batch in tqdm(valid_loader, desc="Valid Loss", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                outputs = model(**batch)
                loss = outputs.loss

            bs = batch["input_ids"].size(0)
            total_loss += loss.item() * bs
            total_count += bs

    return total_loss / max(total_count, 1)

def evaluate_val_acc(model, valid_subset):
    model.eval()
    correct = 0
    total = 0

    for i in tqdm(range(len(valid_subset)), desc="Valid Acc", leave=False):
        row = valid_subset.iloc[i]
        pred = score_options(row)   # 추론용 함수
        gold = str(row["answer"]).strip().lower()

        correct += int(pred == gold)
        total += 1

    return correct / max(total, 1)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [53]:
EPOCHS = 3
GRAD_ACCUM = 8
SAVE_EVERY_STEPS = 300
MAX_GRAD_NORM = 1.0

LR = 7e-5
WEIGHT_DECAY = 0.01

In [54]:
import math
import torch
from transformers import get_cosine_schedule_with_warmup

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
num_training_steps = EPOCHS * num_update_steps_per_epoch
num_warmup_steps = int(num_training_steps * 0.05)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

In [55]:
import torch
from PIL import Image

def score_options(row):
    img = Image.open(row["path"]).convert("RGB")

    q = str(row["question"])
    options = {
        "a": str(row["a"]),
        "b": str(row["b"]),
        "c": str(row["c"]),
        "d": str(row["d"]),
    }

    prompt_messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_mc_prompt(q, options["a"], options["b"], options["c"], options["d"])}
        ]}
    ]

    best_letter = None
    best_score = -1e18

    for letter, answer_text in options.items():
        full_messages = prompt_messages + [
            {"role": "assistant", "content": [{"type": "text", "text": answer_text}]}
        ]

        prompt_text = processor.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True
        )
        full_text = processor.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False
        )

        full_inputs = processor(
            text=[full_text],
            images=[img],
            return_tensors="pt"
        ).to(device)

        prompt_inputs = processor(
            text=[prompt_text],
            images=[img],
            return_tensors="pt"
        ).to(device)

        input_ids = full_inputs["input_ids"]
        attention_mask = full_inputs["attention_mask"]
        pixel_values = full_inputs["pixel_values"]
        image_grid_thw = full_inputs["image_grid_thw"]
        mm_token_type_ids = full_inputs["mm_token_type_ids"]

        prompt_len = prompt_inputs["input_ids"].shape[1]

        labels = input_ids.clone()
        labels[:, :prompt_len] = -100

        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                mm_token_type_ids=mm_token_type_ids,
                labels=labels
            )

        score = -outputs.loss.item()

        if score > best_score:
            best_score = score
            best_letter = letter

    return best_letter

# 비상임


In [ ]:
val_loss = evaluate_val_loss(model, valid_loader, device)
val_acc = evaluate_val_acc(model, valid_subset)

print(f"val_loss: {val_loss:.4f}")
print(f"val_acc : {val_acc:.4f}")

Valid Loss:   0%|          | 0/508 [00:00<?, ?it/s]

/tmp/ipykernel_42057/1366771142.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a46d54ba340>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a46d54ba340>
Traceback (most recent call last):
  Fi

Valid Acc:   0%|          | 0/508 [00:00<?, ?it/s]

val_loss: 0.1372
val_acc : 0.8839


In [ ]:
row_log = {
    "epoch": 1,
    "global_step": global_step,
    "train_loss": avg_train_loss,
    "val_loss": val_loss,
    "val_acc": val_acc,
    "best_val_acc_before": best_val_acc
}
logs.append(row_log)

if val_acc > best_val_acc:
    best_val_acc = val_acc
    save_checkpoint(
        BEST_DIR,
        model,
        processor,
        optimizer,
        scheduler,
        epoch=1,
        global_step=global_step,
        best_val_acc=best_val_acc,
        logs=logs
    )
    print(f"[BEST 갱신] val_acc={val_acc:.4f}")

save_checkpoint(
    LAST_DIR,
    model,
    processor,
    optimizer,
    scheduler,
    epoch=1,
    global_step=global_step,
    best_val_acc=best_val_acc,
    logs=logs
)
save_log(logs, os.path.join(SAVE_ROOT, "train_log.csv"))

[BEST 갱신] val_acc=0.8839


In [ ]:
import os

print("BEST_DIR exists:", os.path.exists(BEST_DIR))
print("LAST_DIR exists:", os.path.exists(LAST_DIR))
print("train_log exists:", os.path.exists(os.path.join(SAVE_ROOT, "train_log.csv")))

print("\nBEST_DIR files:")
print(os.listdir(BEST_DIR))

print("\nLAST_DIR files:")
print(os.listdir(LAST_DIR))

BEST_DIR exists: True
LAST_DIR exists: True
train_log exists: True

BEST_DIR files:
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json', 'trainer_state.pt', 'training_state.json']

LAST_DIR files:
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json', 'trainer_state.pt', 'training_state.json']


In [56]:
from tqdm.auto import tqdm
import torch.nn.utils as nn_utils

logs = []
RESUME = True

if RESUME:
    start_epoch, global_step, best_val_acc, logs = load_checkpoint_if_exists(
        LAST_DIR, model, optimizer, scheduler
    )
else:
    start_epoch, global_step, best_val_acc, logs = 0, 0, -1.0, []

print(f"start_epoch={start_epoch}, global_step={global_step}, best_val_acc={best_val_acc:.4f}")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    train_loss_sum = 0.0
    train_loss_count = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} Train")

    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss
            loss_for_backward = loss / GRAD_ACCUM

        loss_for_backward.backward()

        running_loss += loss.item()
        train_loss_sum += loss.item()
        train_loss_count += 1

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            nn_utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            avg_running = running_loss / GRAD_ACCUM
            pbar.set_postfix({
                "step_loss": f"{avg_running:.4f}",
                "gstep": global_step
            })
            running_loss = 0.0

            if global_step % SAVE_EVERY_STEPS == 0:
                step_save_dir = os.path.join(STEP_DIR, f"step_{global_step}")
                save_checkpoint(
                    step_save_dir,
                    model,
                    processor,
                    optimizer,
                    scheduler,
                    epoch=epoch,
                    global_step=global_step,
                    best_val_acc=best_val_acc,
                    logs=logs
                )

                save_checkpoint(
                    LAST_DIR,
                    model,
                    processor,
                    optimizer,
                    scheduler,
                    epoch=epoch,
                    global_step=global_step,
                    best_val_acc=best_val_acc,
                    logs=logs
                )
                print(f"\n[중간 저장 완료] step={global_step}")

    avg_train_loss = train_loss_sum / max(train_loss_count, 1)

    val_loss = evaluate_val_loss(model, valid_loader, device)
    val_acc = evaluate_val_acc(model, valid_subset)

    row_log = {
        "epoch": epoch + 1,
        "global_step": global_step,
        "train_loss": avg_train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "best_val_acc_before": best_val_acc
    }
    logs.append(row_log)

    print(f"\n[Epoch {epoch+1}]")
    print(f"train_loss: {avg_train_loss:.4f}")
    print(f"val_loss  : {val_loss:.4f}")
    print(f"val_acc   : {val_acc:.4f}")
    print(f"best_acc  : {best_val_acc:.4f}")

    save_checkpoint(
        LAST_DIR,
        model,
        processor,
        optimizer,
        scheduler,
        epoch=epoch + 1,
        global_step=global_step,
        best_val_acc=best_val_acc,
        logs=logs
    )
    save_log(logs, os.path.join(SAVE_ROOT, "train_log.csv"))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(
            BEST_DIR,
            model,
            processor,
            optimizer,
            scheduler,
            epoch=epoch + 1,
            global_step=global_step,
            best_val_acc=best_val_acc,
            logs=logs
        )
        print(f"[BEST 갱신] val_acc={val_acc:.4f}")

    save_checkpoint(
        LAST_DIR,
        model,
        processor,
        optimizer,
        scheduler,
        epoch=epoch + 1,
        global_step=global_step,
        best_val_acc=best_val_acc,
        logs=logs
    )
    save_log(logs, os.path.join(SAVE_ROOT, "train_log.csv"))

print("\n학습 종료")
print(f"최종 best_val_acc: {best_val_acc:.4f}")

체크포인트 로드: /content/drive/MyDrive/recycle_vqa_qwen3/last_checkpoint
start_epoch=1, global_step=571, best_val_acc=0.8839


Epoch 2/3 Train:   0%|          | 0/4565 [00:00<?, ?it/s]


[중간 저장 완료] step=600


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a46d54ba340>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a46d54ba340>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16


[중간 저장 완료] step=900


...

/tmp/ipykernel_42057/1366771142.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):


Valid Acc:   0%|          | 0/508 [00:00<?, ?it/s]


[Epoch 2]
train_loss: 0.2349
val_loss  : 0.1372
val_acc   : 0.8701
best_acc  : 0.8839


Epoch 3/3 Train:   0%|          | 0/4565 [00:00<?, ?it/s]

KeyboardInterrupt: 

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
import torch
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd

def score_options(row):
    img = Image.open(row["path"]).convert("RGB")

    q = str(row["question"])
    options = {
        "a": str(row["a"]),
        "b": str(row["b"]),
        "c": str(row["c"]),
        "d": str(row["d"]),
    }

    prompt_messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_mc_prompt(q, options["a"], options["b"], options["c"], options["d"])}
        ]}
    ]

    best_letter = None
    best_score = -1e18

    for letter, answer_text in options.items():
        full_messages = prompt_messages + [
            {"role": "assistant", "content": [{"type": "text", "text": answer_text}]}
        ]

        prompt_text = processor.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True
        )
        full_text = processor.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False
        )

        full_inputs = processor(
            text=[full_text],
            images=[img],
            return_tensors="pt"
        ).to(device)

        prompt_inputs = processor(
            text=[prompt_text],
            images=[img],
            return_tensors="pt"
        ).to(device)

        input_ids = full_inputs["input_ids"]
        attention_mask = full_inputs["attention_mask"]
        pixel_values = full_inputs["pixel_values"]
        image_grid_thw = full_inputs["image_grid_thw"]
        mm_token_type_ids = full_inputs["mm_token_type_ids"]

        prompt_len = prompt_inputs["input_ids"].shape[1]

        labels = input_ids.clone()
        labels[:, :prompt_len] = -100

        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                mm_token_type_ids=mm_token_type_ids,
                labels=labels
            )

        score = -outputs.loss.item()

        if score > best_score:
            best_score = score
            best_letter = letter

    return best_letter


model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    pred_letter = score_options(row)
    preds.append(pred_letter)

submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds
})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")
print(submission.head())

Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]

Saved /content/submission.csv
              id answer
0  test_0001.jpg      d
1  test_0002.jpg      a
2  test_0003.jpg      c
3  test_0004.jpg      a
4  test_0005.jpg      b


In [ ]:
# 모델 응답 예시
print(output_text)

system
You are a helpful visual question answering assistant. Answer using exactly one letter among a, b, c, or d. No explanation.
user
사진에 보이는 재활용 가능한 아이템은 무엇인가요?
(a) 플라스틱 숟가락
(b) 유리병
(c) 종이 포장지
(d) 금속 캔

정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.
assistant
b
